In [1]:
import os
import json
import pandas as pd
import numpy as np
import pickle

# 1. PATH DATASET
DATASET_DIR = r"C:\Users\Raisa Shahira\Documents\UI\SMT 6\Propen\dataset"
OUTPUT_PATH = r"C:\Users\Raisa Shahira\Downloads\Propen\dataset_cva_hs.pkl"

dataset_cva_hs = []

# 2. LOOP SEMUA FILE
for root, dirs, files in os.walk(DATASET_DIR):
    for filename in files:
        
        # Ambil hanya file metadata
        if not filename.endswith("_meta.json"):
            continue
        
        base_name = filename.replace("_meta.json", "")
        json_path = os.path.join(root, filename)
        
        # 3. LOAD JSON
        try:
            with open(json_path, 'r') as f:
                meta = json.load(f)
        except:
            continue
        
        pathology = meta.get("pathologyKey")
        
        # Filter hanya CVA dan HS
        if pathology not in ["CVA", "HS"]:
            continue
        
        # 4. AMBIL METADATA
        age = meta.get("age")
        gender = meta.get("gender")
        height = meta.get("height")
        weight = meta.get("weight")
        bmi = meta.get("BMI")
        laterality = meta.get("laterality")
        
        deficit_side = meta.get("clinicalDeficitSide")
        if deficit_side is None:
            deficit_side = "None"
        
        fma_score = meta.get("evaluationScoreValue")
        if pathology == "HS" and fma_score is None:
            fma_score = 34.0
        
        uturn_bounds = meta.get("uturnBoundaries", [])
        
        # 5. LOAD DATA SENSOR
        txt_filename = f"{base_name}_processed_data.txt"
        txt_path = os.path.join(root, txt_filename)
        
        if not os.path.exists(txt_path):
            continue
        
        try:
            sensor_df = pd.read_csv(txt_path, sep=r'\s+')
        except:
            continue
        
        # 6. AMBIL KOLOM SENSOR
        try:
            lb_data = sensor_df[['LF_FreeAcc_X', 'LF_FreeAcc_Y', 'LF_FreeAcc_Z',
                                 'LF_Gyr_X', 'LF_Gyr_Y', 'LF_Gyr_Z',
                                 'RF_FreeAcc_X', 'RF_FreeAcc_Y', 'RF_FreeAcc_Z',
                                 'RF_Gyr_X', 'RF_Gyr_Y', 'RF_Gyr_Z']].copy()
        except KeyError:
            continue
        
        # 7. TAMBAH FLAG U-TURN
        lb_data['Uturn_Flag'] = 0
        
        if uturn_bounds and len(uturn_bounds) == 2:
            u_start = int(max(0, uturn_bounds[0]))
            u_end = int(min(len(lb_data), uturn_bounds[1]))
            
            lb_data.iloc[u_start:u_end, lb_data.columns.get_loc('Uturn_Flag')] = 1
        
        # 8. SUSUN DATASET
        dataset_v2 = {
            "subject_id": base_name.rsplit('_', 1)[0],
            "sensor_matrix": lb_data.values.astype(np.float32),  
            "fma_score": float(fma_score),
            "meta_features": {
                "age": age,
                "gender": gender,
                "height": height,
                "weight": weight,
                "BMI": bmi,
                "laterality": laterality,
                "deficit_side": deficit_side
            }
        }
        
        dataset_cva_hs.append(dataset_v2)

# 9. SAVE KE PKL
with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(dataset_cva_hs, f)

print("====================================")
print(f"Ekstraksi selesai!")
print(f"Total trial: {len(dataset_cva_hs)}")
print(f"Dataset disimpan di: {OUTPUT_PATH}")
print("====================================")

Ekstraksi selesai!
Total trial: 488
Dataset disimpan di: C:\Users\Raisa Shahira\Downloads\Propen\dataset_cva_hs.pkl
